<a href="https://colab.research.google.com/github/saranyabalu25/Beyond-the-Screen-Exploring-2024-Thriller-Movies-with-Streamlit/blob/main/project_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install selenium

In [ ]:
!pip install webdriver_manager

In [15]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd
import re

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# 3) Set Chrome options
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Run in headless mode (no UI)
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
# Tell Selenium where the Chromium binary is:
options.binary_location = "/usr/bin/chromium-browser"

# 4) Start the driver
service = Service("/usr/bin/chromedriver")
driver = webdriver.Chrome(service=service, options=options)

# Now you can use driver.get(...) etc.

In [ ]:
# Genres you want to scrape
genres = ["Action","comedy","Animation","sci-fi","documentary"]  # add more if needed


In [ ]:
# Final DataFrame to store all results
final_df = pd.DataFrame()

In [ ]:
for genre in genres:
    url = f"https://www.imdb.com/search/title/?title_type=feature&release_date=2024-01-01,2024-12-31&genres={genre}"
    driver.get(url)
    time.sleep(5)


In [ ]:
def click_load_more():
        try:
            load_more_button = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[2]/div[2]/div[2]/div/span/button/span/span')
            ActionChains(driver).move_to_element(load_more_button).perform()
            load_more_button.click()
            time.sleep(5)
            return True
        except Exception as e:
            print("No more content to load or error:", e)
            return False

In [ ]:
while click_load_more():
        print("Clicked 'Load More' button")


In [ ]:
print("✅ Finished loading all movies for", genre)
    #//*[@id="__next"]/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[2]/div[2]/ul/li[1]/div/div/div/div[1]/div[2]/span/div/span
    #//*[@id="__next"]/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[2]/div[2]/ul/li[1]/div/div/div
    #//*[@id="__next"]/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[2]/div[2]/ul/li[1]./div/div/div/div[1]/div[2]/div[1]
titles = []
ratings = []
votings = []
durations= []

✅ Finished loading all movies for documentary


In [ ]:
movie_items = driver.find_elements(By.XPATH, '//*[@id="__next"]/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[2]/div[2]/ul/li')

In [ ]:
for movie_item in movie_items:
        try:
            title = movie_item.find_element(By.XPATH, './div/div/div/div[1]/div[2]/div[1]/a/h3').text
            rating = movie_item.find_element(By.XPATH, './div/div/div/div[1]/div[2]/span/div/span/span[1]').text
            voting = movie_item.find_element(By.XPATH, './div/div/div/div[1]/div[2]/span/div/span/span[2]').text
            duration = movie_item.find_element(By.XPATH, './div/div/div/div[1]/div[2]/div[2]/span[2]').text

            titles.append(title)
            ratings.append(rating)
            votings.append(voting)
            durations.append(duration)

        except Exception as e:
            print(f"Error extracting data for a movie: {e}")
            continue

In [ ]:
df = pd.DataFrame({
        'Title': titles,
        'Rating': ratings,
        'Votes': votings,
        'Duration': durations,
        'Genre': genre
    })

In [ ]:
for genre in df['Genre'].unique():
    if pd.isna(genre):
        continue
    genre_df = df[df['Genre'].str.contains(genre.split(",")[0], na=False)]
    genre_name = genre.split(",")[0].strip()

    # Clean Title and Votes
    genre_df['Title'] = genre_df['Title'].str.replace(r'^\d+\.\s*', '', regex=True)
    genre_df['Votes'] = genre_df['Votes'].str.replace(r'[\(\)]', '', regex=True)

    # Save genre-wise CSV
    genre_df.to_csv(f"{genre_name}_2024_movies_og2.csv", index=False)

    # If you want to build a final_df
    final_df = pd.concat([final_df, genre_df], ignore_index=True)

In [ ]:
# Save combined CSV

final_df.to_csv("all_genres_2024_movies_og2.csv", index=False)
print("\n All genres saved to all_genres_2024_movies.csv")

driver.quit()


 All genres saved to all_genres_2024_movies.csv


Step 2: Cleaning and Transforming Movie Data This section includes helper functions to clean and transform raw data, such as converting durations and vote counts into usable formats.

In [1]:
import re

# Function to convert duration to total minutes as int
def convert_duration_to_minutes(duration):
    duration = duration.lower().strip()
    hours = minutes = 0
    hr_match = re.search(r'(\d+)\s*h', duration)
    min_match = re.search(r'(\d+)\s*m', duration)
    if hr_match:
        hours = int(hr_match.group(1))
    if min_match:
        minutes = int(min_match.group(1))
    return hours * 60 + minutes

In [2]:
# Function to convert vote strings like "53K" to integer
def convert_votes_to_int(votes):
    votes = votes.strip().upper()
    if 'K' in votes:
        return int(float(votes.replace('K', '')) * 1000)
    elif 'M' in votes:
        return int(float(votes.replace('M', '')) * 1000000)
    return int(votes)




In [3]:
try:
    duration = driver.find_element(By.XPATH, "//li[@data-testid='title-techspec_runtime']//div").text
except:
    duration = "N/A"

try:
    votes = driver.find_element(By.CSS_SELECTOR, "div.sc-89bddf1f-3.iKuuud span").text
except:
    votes = "N/A"

In [ ]:
movie_data.append({
    "Movie Name": movie_names[i],
    "Genre": genre,
    "Rating": rating,
    "Duration": duration,
    "Votes": votes
})